# Multi-Entity Biomedical NER — Experiment Runner

One-click notebook to launch every training configuration (1..6) with one
or more seeds, then collate the per-seed evaluation JSONs into mean ± std
tables.

This notebook calls the training functions **directly** in the kernel's
Python process — no `bash`, no `subprocess`, no environment crossing. Make
sure the kernel uses the Python where `requirements.txt` is installed.

Bash scripts under `scripts/` remain available for headless multi-seed
runs on Vast.ai / SSH sessions; the notebook does not invoke them.

## 1. Setup — run once per session

In [ ]:
import os
import sys
from pathlib import Path

# Ensure we work from the project root so relative paths in YAML resolve.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')

import torch, transformers
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Transformers: {transformers.__version__}')

from src.config import load_config
from src.training.trainer_single import train_single
from src.training.trainer_sequential import train_sequential
from src.training.trainer_joint import train_joint
from src.utils.aggregate import aggregate_seeds
from src.utils.logging import configure_logging

# One log file per notebook session.
configure_logging(PROJECT_ROOT / 'outputs' / 'logs' / 'notebook', run_name='session')

## 2. (Optional) Install requirements

Skip if your venv already has the deps. PyTorch should be installed
separately first (CPU or matching CUDA build) before running this cell.

In [ ]:
%pip install -r requirements.txt

## 3. Verify dataset files exist

In [ ]:
for rel in [
    'dataset/bc5cdr/bc5cdr_train.jsonl',
    'dataset/bc5cdr/bc5cdr_validation.jsonl',
    'dataset/bc5cdr/bc5cdr_test.jsonl',
    'dataset/pubmed/pubmed_scrapping.jsonl',
    'dataset/pubmed/pubmed_test.jsonl',
]:
    p = Path(rel)
    print(f"  {rel}: {'OK' if p.exists() else 'MISSING'}  ({p.stat().st_size if p.exists() else 0:,} bytes)")

## 4. Pick seeds and helper

Default is three seeds for paper-grade mean ± std reporting. Change `SEEDS`
once here and every config cell below will use the new value.

Set `SMOKE_TEST = True` for a quick plumbing check (16 training sentences,
1 epoch). Set it back to `False` for full runs.

In [ ]:
SEEDS = [42, 1337, 2024]
SMOKE_TEST = False  # flip to True for a quick pipeline check

TRAINER_BY_STRATEGY = {
    'single': train_single,
    'sequential': train_sequential,
    'joint_uniform': train_joint,
    'joint_noise_aware': train_joint,
}

def run_config(config_path: str, seeds=None, smoke_test=None) -> None:
    """Train one experiment across the given seeds, then aggregate."""
    seeds = seeds if seeds is not None else SEEDS
    smoke_test = smoke_test if smoke_test is not None else SMOKE_TEST
    cfg = load_config(config_path)
    trainer_fn = TRAINER_BY_STRATEGY[cfg.training.strategy]
    print('=' * 70)
    print(f'  {cfg.experiment.name}  |  strategy={cfg.training.strategy}  |  seeds={seeds}')
    print('=' * 70)
    for seed in seeds:
        print(f'\n>>> {cfg.experiment.name} — seed {seed}\n')
        trainer_fn(config=cfg, seed=seed, smoke_test=smoke_test)
    print(f'\n>>> Aggregating {cfg.experiment.name}')
    aggregate_seeds(cfg.experiment.name, base_dir=str(PROJECT_ROOT / 'outputs'))

print(f'Using SEEDS={SEEDS}, SMOKE_TEST={SMOKE_TEST}')

## 5. Config 1 — BERT-base + BC5CDR (lower-bound baseline)

In [ ]:
run_config('configs/config_1_bert_base.yaml')

## 6. Config 2 — BioBERT + BC5CDR

In [ ]:
run_config('configs/config_2_biobert.yaml')

## 7. Config 3 — PubMedBERT + BC5CDR

In [ ]:
run_config('configs/config_3_pubmedbert.yaml')

## 8. Config 4 — Sequential (BC5CDR → silver)

Trains 9-tag head; phase 1 sees only Chem/Dis, phase 2 lifts Virus/Gene
from the silver corpus. Forgetting score on Chem/Dis is logged automatically.

In [ ]:
run_config('configs/config_4_sequential.yaml')

## 9. Config 5 — Joint training (uniform sample weights)

In [ ]:
run_config('configs/config_5_joint_uniform.yaml')

## 10. Config 6 — Joint training (noise-aware, silver=0.3)

In [ ]:
run_config('configs/config_6_joint_noise_aware.yaml')

## 11. View aggregated results

Each `run_config(...)` call already wrote `aggregated_results.{json,md}`
into `outputs/results/<config>/`. The cell below renders every available
Markdown table for quick side-by-side comparison.

In [ ]:
from IPython.display import Markdown, display
for results_dir in sorted(Path('outputs/results').glob('config_*')):
    md_file = results_dir / 'aggregated_results.md'
    if md_file.exists():
        display(Markdown(md_file.read_text(encoding='utf-8')))
    else:
        print(f'[no aggregated_results.md yet] {results_dir.name}')

## 12. (Optional) Inference smoke test

After Config 4 finishes, run inference on the PubMed test set using the
phase-2 checkpoint. Change the path to whichever checkpoint you want.

In [ ]:
from src.inference.predictor import NERPredictor
import json

CHECKPOINT = 'outputs/checkpoints/config_4_sequential/seed_42/phase2/best_model'
INPUT_JSONL = 'dataset/pubmed/pubmed_test.jsonl'
OUTPUT_JSONL = 'outputs/predictions/config_4_pubmed_test.jsonl'

predictor = NERPredictor(CHECKPOINT, batch_size=16)
records = [json.loads(line) for line in Path(INPUT_JSONL).open('r', encoding='utf-8') if line.strip()]
preds = predictor.predict_records(records, token_field='tokens', id_field='sentence_id')

Path(OUTPUT_JSONL).parent.mkdir(parents=True, exist_ok=True)
with Path(OUTPUT_JSONL).open('w', encoding='utf-8') as fp:
    for record in preds:
        fp.write(json.dumps(record, ensure_ascii=False) + '\n')
print(f'Wrote {len(preds)} predictions to {OUTPUT_JSONL}')
print('First record:', json.dumps(preds[0], indent=2)[:600])